# Changes Compared to `preprocessing.ipynb`

This notebook is a copy of `preprocessing.ipynb` (commit `7627027`) with the following changes. All other known issues of the original are intentionally left untouched for now.

1. **Data download from Kaggle.** A new *Downloading the Data* section fetches `train.csv` / `test.csv` from the [`music-genre` competition](https://www.kaggle.com/competitions/music-genre/data) via the Kaggle API, so the notebook no longer depends on files being placed in `data/` by hand.
2. **NLTK resources are downloaded** in the import cell (`stopwords`, `wordnet`, `punkt`, ...), so the notebook runs in a fresh environment.
3. **Fixed stopword removal.** The cleaning functions now come from `transformers/pipeline_transformers.py`, a copy of `utils/pipeline_v2.py` with these fixes (the original module is unchanged, so `preprocessing.ipynb` behaves as before):
   - Stopwords were removed **twice**: once in `regex_cleaner`, *before* contractions were expanded, and again in `main_pipeline`. They are now removed **once**, in `main_pipeline`, after `contractions.fix` and tokenisation.
   - **Negations (*no*, *nor*, *not*) are kept.** NLTK's stopword list contains them, so lines like *"I don't love you"* were reduced to *"love"*, which inverts their meaning for sentiment analysis.
   - **Curly apostrophes (’) are normalised to `'`.** They fell inside the punctuation pattern, so *"don’t"* was split into *"don t"* before it could be expanded, and both halves were then dropped as stopwords.

   | Input | Before | After |
   |---|---|---|
   | I don't love you, no more | `love` | `not love no` |
   | I don’t wanna cry tonight | `want cry tonight` | `not want cry tonight` |
   | You can't stop me, I'm not afraid | `stop afraid` | `not stop not afraid` |

4. **Fixed `NameError` in the first split** (*Reading and Splitting the Training Data*): `stratify=df['tag']` referenced an undefined variable `df`; it now stratifies on `df_train['tag']`, the DataFrame being split.
5. **Fixed `NameError` in class balancing**: `balance_classes(cleaned_df, ...)` referenced an undefined variable `cleaned_df`. It now balances the training split (`X_train` + `y_train`), so the validation set keeps the real genre distribution, and exposes the result as `X_train_balanced` / `y_train_balanced`.
6. **Fixed `FileNotFoundError` in GloVe training**: the vocabulary was read from `vocab.parquet` instead of `glove_dir / "vocab.parquet"`, where the previous cell saves it.
7. **Separate transformer input.** The cell that saved the tokenized data a second time to `data-transformers/` (identical to the `data-w2v/` copy) is replaced by a *Transformer Input* section that saves lightly cleaned lyrics plus `id` and metadata to `data-transformers/transformers_train.parquet` and `transformers_test.parquet`.
8. **Validation split by song id.** A stratified 25% validation split of the raw training data is saved as `data/validation_ids.parquet`, so all models can be compared on the same validation songs.
9. **Long lyrics handling** (new *Long Lyrics and the 512-Token Limit* section, functions in `transformers/long_lyrics.py`): measures how many songs exceed the 512-token limit and how many lines repeat within a song, then adds two columns to the transformer files: `lyrics_classification` (repeated lines removed, head + tail truncation for songs still too long) and `lyrics_sentiment` (full lyrics, songs over the limit split into two parts at the stanza break closest to the middle).
10. **Moved to the `transformers/` folder** together with the other transformer files (`transformers.ipynb`, `transformer_evaluation.ipynb`, `pipeline_transformers.py`, `long_lyrics.py`). The data paths now point to `../data`, and the project root is added to `sys.path` for the shared `utils` package.


The environment is described in `requirements.txt` (conda env `text_mining_project`).

# Preprocessing

In [24]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
from mittens import GloVe
import nltk
for pkg in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
import sys
sys.path.append("..")  # project root, for the shared utils package
import pipeline_transformers
from utils import funcs
from transformers import pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from collections import defaultdict

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

subfolders = ["data-GloVe", "data-transformers", "data-RF", "data-w2v"]

for sub in subfolders:
    path = Path(data_dir) / sub
    path.mkdir(parents=True, exist_ok=True)

glove_dir=data_dir / "data-GloVe"
tran_dir=data_dir / "data-transformers"
rf_dir=data_dir / "data-RF"
w2v_dir=data_dir / "data-w2v"

## Downloading the Data

The raw data is downloaded from the [Kaggle competition page](https://www.kaggle.com/competitions/music-genre/data) with the official Kaggle API and extracted into `data/`. The download is skipped if the files are already present.

This requires a Kaggle API token (`kaggle auth login`, or a token from *Kaggle → Settings → API*) and having accepted the competition rules on the competition page.

**Note:** an old `~/.kaggle/kaggle.json` API key takes precedence over a `kaggle auth login` session. If that key is expired, the download fails with `401 Unauthorized` even after logging in; rename or remove `kaggle.json` to fix it.

In [26]:
import zipfile

if not (data_dir / "train.csv").exists():
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.competition_download_files("music-genre", path=data_dir, quiet=False)
    with zipfile.ZipFile(data_dir / "music-genre.zip") as z:
        z.extractall(data_dir)
    (data_dir / "music-genre.zip").unlink()

sorted(p.name for p in data_dir.iterdir())

['.DS_Store',
 'balanced_train_df.parquet',
 'data-GloVe',
 'data-RF',
 'data-transformers',
 'data-w2v',
 'sample_submission.csv',
 'test.csv',
 'train.csv']

## Reading and Splitting the Training Data

To optimize data preprocessing in the pipeline, we load the raw data into a **Polars DataFrame**, leveraging its multi-threaded engine to handle large files faster and more efficiently than pandas. 

To manage local memory constraints, the dataset is split into two equal partitions (`sample_df1` and `sample_df2`) and processed sequentially.

In [27]:
df_train = pl.read_csv(data_dir / 'train.csv')
df_test = pl.read_csv(data_dir / 'test.csv')

In [28]:
sample_df1, sample_df2 = train_test_split(
    df_train,
    train_size=0.5,
    stratify=df_train['tag'],
    random_state=80085
)

### Lyrics Preprocessing Overview

Each partition is mapped through `main_pipeline` via Polars' `map_elements` to clean and standardize the `lyrics` column:

- **Text Cleaning & Normalization:** Strips song metadata tags (e.g., *[Chorus]*, *[Verse]*), URLs, brackets, and punctuation, while expanding contractions.
- **Filtering & Tokenization:** Tokenizes text into individual words, discards non-alphabetic tokens, and removes standard English stopwords (once, after contractions are expanded, keeping the negations *no*, *nor*, *not*) as well as common musical filler vocalizations (e.g., *"oh"*, *"yeah"*, *"la"*).
- **Linguistic Standardization:** Converts diacritics to ASCII, applies lowercasing, and lemmatizes tokens across multiple parts of speech.
- **Output Schema:** With `tokenized_output=True`, each song's lyrics are transformed into a list of cleaned tokens (`pl.List(pl.String)`) stored under `cleaned_lyrics`.

Once both batches finish processing, they are combined back into a single unified dataset (`cleaned_df`) using vertical concatenation (`pl.concat`).

In [29]:
from pipeline_transformers import main_pipeline

sample_df1 = sample_df1.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

sample_df2 = sample_df2.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String) #if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

cleaned_df_train = pl.concat([sample_df1, sample_df2])

cleaned_df_test = df_test.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics"))

## Checkpointing & Class Distribution Overview

To preserve our preprocessed data and avoid redundant recomputation, we export the DataFrame to **Parquet format** and reload it. Parquet provides efficient column-oriented compression and natively preserves complex nested schemas, such as columns containing lists of strings (`pl.List(pl.String)`).

An initial inspection of the target genre distribution reveals a notable class imbalance, with **Rock**, **Pop**, and **Rap** making up the substantial majority of the dataset. Downstream modeling and evaluation strategies will need to account for this skew.

### Transformer Input

Transformer models read words in context and use their own subword tokenizer, so they get the lyrics with only **light cleaning** instead of the tokenized version above: stopwords, punctuation, contractions, casing and line breaks are kept, since removing them changes the meaning of a line (e.g. *"Cause it's over now"* consists only of stopwords). Only noise is removed: square-bracket annotations (section tags such as *[Verse 1: Artist]*, *[Chorus]* and Genius' *[?]* marker for unknown lyrics), URLs and invisible characters, plus redundant whitespace. Lyrics in parentheses are kept, as they are usually sung backing vocals.

The saved files (end of the *Long Lyrics* section) keep `id` and the song metadata (`year`, `views`, ...) needed for the sentiment analysis and the Kaggle submission.

In [30]:
import re

square_brackets = re.compile(r"\[[^\]\n]*\]")
url_pattern = re.compile(r"(?:https?://|www\.)\S+")
invisible_chars = re.compile(r"[​﻿]")
spaces = re.compile(r"[ \t]+")
extra_blank_lines = re.compile(r"\n{3,}")

def light_clean(text):
    text = text.replace("’", "'").replace("‘", "'")
    text = square_brackets.sub("", text)
    text = url_pattern.sub("", text)
    text = invisible_chars.sub("", text)
    text = spaces.sub(" ", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    return extra_blank_lines.sub("\n\n", text).strip()   # keeps one blank line between stanzas

metadata = ['id', 'title', 'artist', 'year', 'views', 'features']

tr_train = (df_train.with_columns(pl.col('lyrics').map_elements(light_clean, return_dtype=pl.String))
                    .select(metadata + ['tag', 'lyrics']))
tr_test = (df_test.with_columns(pl.col('lyrics').map_elements(light_clean, return_dtype=pl.String))
                  .select(metadata + ['lyrics']))

### Validation Split by Song ID

The transformer models are evaluated on a fixed, stratified 25% validation split. The split is made on the song `id` of the raw training data, so it is reproducible (same `train.csv` and seed give the same songs) and every model, whatever its preprocessing, can be evaluated on exactly the same validation songs by filtering on these ids.

In [31]:
train_ids, val_ids = train_test_split(
    df_train['id'].to_numpy(), test_size=0.25, stratify=df_train['tag'].to_numpy(), random_state=80085
)
pl.DataFrame({'id': val_ids}).write_parquet(data_dir / 'validation_ids.parquet')
print(f"train: {len(train_ids)} songs, validation: {len(val_ids)} songs")

train: 101225 songs, validation: 33742 songs


### Long Lyrics and the 512-Token Limit

Transformer encoders such as RoBERTa read at most 512 tokens, i.e. 510 tokens of lyrics plus two special tokens; anything beyond is cut off. Token counts are measured with the tokenizer of `distilroberta-base`, which uses the same byte-level BPE vocabulary as RoBERTa-based sentiment models (e.g. `cardiffnlp/twitter-roberta-base-sentiment-latest`), so the counts hold for both tasks. Only the tokenizer files are downloaded here, no model weights.

The two tasks handle long songs differently:
- **Genre classification:** repeated lines (mostly choruses) are removed first, as they add length without new content. Songs that are still too long are reduced to their first 128 and last 381 tokens (*head + tail*), the truncation strategy that performed best for long texts in Sun et al. (2019).
- **Sentiment analysis:** the sentiment of the *whole* song is of interest and repetition is part of a song's emotional emphasis, so the full lyrics are kept. Songs that are too long are split into two parts at the stanza break closest to the middle; each part is scored separately and the song's score is the token-length-weighted average of both parts (`average_part_scores`).

*Sun, C., Qiu, X., Xu, Y., & Huang, X. (2019). How to Fine-Tune BERT for Text Classification? In* Chinese Computational Linguistics (CCL 2019)*, LNCS 11856, 194–206. Springer.*

In [32]:
from transformers import AutoTokenizer
from long_lyrics import (MAX_TOKENS, token_counts, repeated_lines, remove_repeated_lines,
                         head_tail, split_in_two)

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

n_tokens_train = token_counts(tr_train['lyrics'].to_list(), tokenizer)
n_tokens_test = token_counts(tr_test['lyrics'].to_list(), tokenizer)
print(f"songs over {MAX_TOKENS} tokens: {(n_tokens_train > MAX_TOKENS).mean():.1%} (train), "
      f"{(n_tokens_test > MAX_TOKENS).mean():.1%} (test)")

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (541 > 512). Running this sequence through the model will result in indexing errors


songs over 510 tokens: 26.2% (train), 26.3% (test)


#### Repeated Lines

A line counts as repeated if the same line (ignoring casing and surrounding whitespace) already appeared earlier in the same song. Only these repeats are removed; the first occurrence of every line is kept.

In [33]:
line_stats = np.array([repeated_lines(t) for t in tr_train['lyrics']])
repeats = tr_train.select('tag').with_columns(lines=line_stats[:, 0], repeated=line_stats[:, 1])

print(f"{repeats['repeated'].sum():,} of {repeats['lines'].sum():,} lines "
      f"({repeats['repeated'].sum() / repeats['lines'].sum():.1%}) repeat an earlier line of the same song")
print(f"{(repeats['repeated'] > 0).mean():.1%} of songs contain at least one repeated line")

(repeats.group_by('tag')
        .agg(share_of_lines_repeated=(pl.col('repeated').sum() / pl.col('lines').sum()).round(3),
             median_repeated_lines=pl.col('repeated').median())
        .sort('tag'))

1,417,568 of 5,628,345 lines (25.2%) repeat an earlier line of the same song
81.3% of songs contain at least one repeated line


tag,share_of_lines_repeated,median_repeated_lines
str,f64,f64
"""country""",0.286,7.0
"""misc""",0.08,0.0
"""pop""",0.313,8.0
"""rap""",0.196,8.0
"""rb""",0.309,12.0
"""rock""",0.284,7.0


In [34]:
deduped_train = [remove_repeated_lines(t) for t in tr_train['lyrics']]
deduped_test = [remove_repeated_lines(t) for t in tr_test['lyrics']]
n_dedup_train = token_counts(deduped_train, tokenizer)
n_dedup_test = token_counts(deduped_test, tokenizer)

overflow = tr_train.select('tag').with_columns(before=n_tokens_train > MAX_TOKENS, after=n_dedup_train > MAX_TOKENS)
print(f"songs over {MAX_TOKENS} tokens: {overflow['before'].mean():.1%} -> {overflow['after'].mean():.1%} "
      f"after removing repeated lines")
overflow.group_by('tag').agg(pl.col('before', 'after').mean().round(3)).sort('tag')

songs over 510 tokens: 26.2% -> 16.1% after removing repeated lines


tag,before,after
str,f64,f64
"""country""",0.065,0.02
"""misc""",0.402,0.384
"""pop""",0.116,0.035
"""rap""",0.599,0.429
"""rb""",0.303,0.104
"""rock""",0.054,0.015


#### Classification Input: Head + Tail

The songs that are still over the limit after removing repeated lines are truncated to their first 128 and last 381 tokens, joined by a line break.

In [35]:
tr_train = tr_train.with_columns(lyrics_classification=pl.Series(
    [head_tail(t, tokenizer) if n > MAX_TOKENS else t for t, n in zip(deduped_train, n_dedup_train)]))
tr_test = tr_test.with_columns(lyrics_classification=pl.Series(
    [head_tail(t, tokenizer) if n > MAX_TOKENS else t for t, n in zip(deduped_test, n_dedup_test)]))

n_final = token_counts(tr_train['lyrics_classification'].to_list(), tokenizer)
print(f"head + tail applied to {(n_dedup_train > MAX_TOKENS).mean():.1%} of training songs; "
      f"longest classification input: {n_final.max()} tokens")

head + tail applied to 16.1% of training songs; longest classification input: 510 tokens


#### Sentiment Input: Two Parts

Songs over the limit are split into two parts (see `split_in_two`). Songs longer than twice the limit still have parts over 510 tokens; these parts are truncated when scored.

In [36]:
tr_train = tr_train.with_columns(lyrics_sentiment=pl.Series(
    [split_in_two(t, tokenizer) if n > MAX_TOKENS else [t] for t, n in zip(tr_train['lyrics'], n_tokens_train)]))
tr_test = tr_test.with_columns(lyrics_sentiment=pl.Series(
    [split_in_two(t, tokenizer) if n > MAX_TOKENS else [t] for t, n in zip(tr_test['lyrics'], n_tokens_test)]))

split_songs = tr_train.filter(pl.col('lyrics_sentiment').list.len() == 2)
part_lengths = token_counts(split_songs['lyrics_sentiment'].explode().to_list(), tokenizer).reshape(-1, 2)
print(f"{len(split_songs) / len(tr_train):.1%} of training songs split into two parts")
print(f"median part lengths: {np.median(part_lengths, axis=0)} tokens; "
      f"songs with a part still over {MAX_TOKENS} tokens: {(part_lengths.max(axis=1) > MAX_TOKENS).sum()}")

26.2% of training songs split into two parts
median part lengths: [349. 351.] tokens; songs with a part still over 510 tokens: 7135


In [37]:
tr_train.write_parquet(tran_dir / 'transformers_train.parquet')
tr_test.write_parquet(tran_dir / 'transformers_test.parquet')
tr_train.select('id', 'tag', 'lyrics_classification', 'lyrics_sentiment').head(3)

id,tag,lyrics_classification,lyrics_sentiment
i64,str,str,list[str]
535805,"""rock""","""When you said you loved me Did…","[""When you said you loved me Did you mean it then Did you change your mind one day Or was it all pretend Cause it's over now See you walk away See you walk away Seems like nothing's gonna change""]"
7519483,"""rap""","""Kid Naruto Rap Everybody want…","[""Kid Naruto Rap Everybody wants you to hurt Everybody wants you to cry with your face In the dirt Gotta put in time, gotta work Everybody's praying that I fall But I gotta make it work Everybody wants you to hurt Everybody wants you to cry with your face In the dirt Gotta put in time, gotta work Everybody's praying that I fall But I gotta make it Hoping it might work out one day Wrong Do it yourself or you might as well fall Gotta do the grind If you never wanna fall Better Aim for the top or you never won at all As a kid I had nobody Grew up parentless Had to go and be funny Had to hide the pain some way On my own path That's my ninja way Headed for the top On a day to day They treat me like outcast Got Stay out the way Gotta hide emotions Gotta hidе the pain This demon fox inside Keeps on driving mе insane Everybody wants you to hurt Everybody wants you to cry with your face In the dirt Gotta put in time, gotta work Everybody's praying that I fall But I gotta make it work"", ""Everybody wants you to hurt Everybody wants you to cry with your face In the dirt Gotta put in time, gotta work Everybody's praying that I fall But I gotta make it Ima be the fucking goat don't need no confirmation You can't replicate my jutsu you ain't as amazing That nigga swear he up to par stop the fabricating I told y'all I'm a sage and y'all ain't got the proper training I ain't have no friends or family helping at all I got demons all Inside of me I'm telling you I'm scarred I had teachers left and right telling me I won't make it far When you different everybody only focus on your flaws Everybody wants you to hurt Everybody wants you to cry with your face In the dirt Gotta put in time, gotta work Everybody's praying that I fall But I gotta make it work Everybody wants you to hurt Everybody wants you to cry with your face In the dirt Gotta put in time, gotta work Everybody's praying that I fall But I gotta make it work Everybody wants you to hurt Everybody wants you to cry with your face In the dirt Gotta put in time, gotta work Everybody's praying that I fall But I gotta make it""]"
4892808,"""pop""","""People tell me I've changed I …","[""People tell me I've changed I find it hard to explain Maybe I feel the same Don't know how to make it better, better again They say I must be happy now Got it all figured out I guess it's gotta hurt sometimes This is what I asked for This is what I asked for, right? All the things that I've done Countless times I've been wrong Maybe that's how you learn But I just wanna make it better, better again They say I must be happy now Got it all figured out I guess it's gotta hurt sometimes This is what I asked for This is what I asked for, right? This is what I asked for This is what I asked for, right?""]"


In [38]:
cleaned_df_train.select(['tag', 'cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')
cleaned_df_test.select(['cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_test_df_tokenized.parquet')

In [39]:
cleaned_df_new = pl.read_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')

In [40]:
cleaned_df_new['tag'].value_counts()

tag,count
str,u32
"""pop""",55742
"""rock""",25332
"""country""",3466
"""misc""",5640
"""rap""",38584
"""rb""",6203


In [41]:
X = cleaned_df_new.select('cleaned_lyrics')
y = cleaned_df_new.select('tag')

In [42]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=80085, stratify=y
)

## Class Balancing via Controlled Downsampling

To resolve the genre distribution skew observed during EDA, we apply `balance_classes` to the training set. Rather than collapsing every category strictly to the minority class count, the routine samples each genre into a bounded range around the target threshold to retain class diversity. The balanced records are then split back into distinct `X` and `y` structures, before being passed through `the test_train_split` dropping the temporary helper column.
4. **Saving the Dataset** For posterity the dataset is saved to Parquet.

In [43]:
from utils.funcs import balance_classes

# Balance only the training split: the validation set must keep the real genre distribution
balanced_df = balance_classes(X_train.hstack(y_train), class_col="tag", seed=80085)

X_train_balanced = balanced_df.select("cleaned_lyrics")
y_train_balanced = balanced_df.select("tag")

balanced_df["tag"].value_counts()

tag,count
str,u32
"""rap""",3061
"""country""",2600
"""pop""",2583
"""rb""",2166
"""misc""",2226
"""rock""",3105


In [44]:
balanced_df.write_parquet(data_dir / 'balanced_train_df.parquet')

## Dataset Branching: Co-Occurrence Matrix Generation

Rather than restricting downstream modeling to a single architecture, we branch our data preparation to support varied embedding strategies, ranging from traditional distributional methods (Word2Vec, GloVe) to deep sequence models (Transformers).

### Co-Occurrence Computation
Count-based embeddings require pairwise co-occurrence frequencies across the corpus. Computing a full co-occurrence matrix naively can trigger SciPy out-of-memory errors due to quadratic vocabulary growth. 

We run `cooccurrence_matrix_generator` to slice matrix operations into manageable chunks, prune self-co-occurrences, and persist the output as a compact Parquet file (`data/cooccurrence_matrix.parquet`).

In [45]:
from utils.funcs import cooccurrence_matrix_generator

df_cooccurrence = cooccurrence_matrix_generator(X_train, "cleaned_lyrics")
df_cooccurrence.write_parquet(glove_dir / "cooccurrence_matrix.parquet")

Computing pairs in blocks: 100%|██████████| 31/31 [00:12<00:00,  2.55it/s]


Assembling final co-occurrence table...


In [46]:
cooc_matrix = pl.scan_parquet(glove_dir / 'cooccurrence_matrix.parquet')
# how many distinct words? (cheap, works on the integer codes)
print(cooc_matrix.select(pl.col("word_1").n_unique()).collect(engine="streaming"))

# the 100k most frequent words
top = (cooc_matrix.group_by("word_1").agg(pl.col("count").sum())
         .sort("count", descending=True).head(100_000)
         .collect(engine="streaming")["word_1"].cast(pl.String))
vocab = top.sort()
enum = pl.Enum(vocab)

# filter, encode to ids, and write to disk, all in chunks
(cooc_matrix.filter(pl.col("count") > 1)                          # drops most of the noise first
   .with_columns(pl.col("word_1", "word_2").cast(pl.String))
   .filter(pl.col("word_1").is_in(vocab) & pl.col("word_2").is_in(vocab))
   .select(pl.col("word_1").cast(enum).to_physical().cast(pl.Int32).alias("i"),
           pl.col("word_2").cast(enum).to_physical().cast(pl.Int32).alias("j"),
           pl.col("count").cast(pl.Float32).alias("x"))
   .sink_parquet(glove_dir / "pairs.parquet"))

vocab.to_frame("word").write_parquet(glove_dir / "vocab.parquet")
print("pairs kept:", pl.scan_parquet(glove_dir / "pairs.parquet").select(pl.len()).collect())

shape: (1, 1)
┌────────┐
│ word_1 │
│ ---    │
│ u32    │
╞════════╡
│ 150998 │
└────────┘


/var/folders/zc/tkfty0q10mx3n2n85rgs661h0000gn/T/ipykernel_32634/3803949885.py:19: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .sink_parquet(glove_dir / "pairs.parquet"))


pairs kept: shape: (1, 1)
┌──────────┐
│ len      │
│ ---      │
│ u32      │
╞══════════╡
│ 91443070 │
└──────────┘


In [47]:
pairs = pl.read_parquet(glove_dir / "pairs.parquet")
i, j, x = pairs["i"].to_numpy(), pairs["j"].to_numpy(), pairs["x"].to_numpy()
del pairs

idx = np.arange(len(x), dtype=np.int32)
np.random.default_rng(0).shuffle(idx)
np.save(glove_dir / "i.npy", i[idx]); np.save(glove_dir / "j.npy", j[idx]); np.save(glove_dir / "x.npy", x[idx])
del i, j, x, idx

In [48]:
vocab = pl.read_parquet(glove_dir / "vocab.parquet")["word"]
from utils.GloVe import GloVe
glove = GloVe(vocab, dim=25).fit(glove_dir / "i.npy", glove_dir / "j.npy", glove_dir / "x.npy", epochs=20)
print(glove.most_similar("love"))
glove.save(glove_dir / "glove_25d")

epoch 1/20  loss 0.1645
epoch 2/20  loss 0.0217
epoch 3/20  loss 0.0197
epoch 4/20  loss 0.0164
epoch 5/20  loss 0.0140
epoch 6/20  loss 0.0130
epoch 7/20  loss 0.0125
epoch 8/20  loss 0.0121
epoch 9/20  loss 0.0118
epoch 10/20  loss 0.0116
epoch 11/20  loss 0.0114
epoch 12/20  loss 0.0112
epoch 13/20  loss 0.0111
epoch 14/20  loss 0.0109
epoch 15/20  loss 0.0108
epoch 16/20  loss 0.0107
epoch 17/20  loss 0.0105
epoch 18/20  loss 0.0104
epoch 19/20  loss 0.0103
epoch 20/20  loss 0.0102
[('let', 0.996), ('feel', 0.995), ('come', 0.994), ('take', 0.993), ('know', 0.993), ('leave', 0.993), ('one', 0.993), ('never', 0.993)]


In [49]:
vectors, vocab = GloVe.load_vectors(glove_dir / "glove_25d")
index = {word: k for k, word in enumerate(vocab)}

In [50]:
def song_vector(tokens, vectors, index, idf=None):
    ids = [index[t] for t in tokens if t in index]
    if not ids:
        return np.zeros(vectors.shape[1], dtype=np.float32)
    if idf is None:
        return vectors[ids].mean(axis=0)
    weights = np.array([idf[t] for t in tokens if t in index], dtype=np.float32)
    return np.average(vectors[ids], axis=0, weights=weights)

In [51]:
X_train_glove = np.vstack([song_vector(t, vectors, index) for t in X_train['cleaned_lyrics']])
X_val_glove = np.vstack([song_vector(t, vectors, index) for t in X_val['cleaned_lyrics']])

print(X_train_glove.shape)

(101225, 25)


In [52]:
def save_glove_csv(X_glove, y, path, label_name="genre"):
    y_arr = y.to_numpy() if hasattr(y, "to_numpy") else np.asarray(y)
    y_arr = y_arr.ravel()                      # flattens a one column frame to 1D

    assert len(X_glove) == len(y_arr), "features and labels have different lengths"

    cols = [f"dim_{k}" for k in range(X_glove.shape[1])]
    df = pl.DataFrame(X_glove, schema=cols, orient="row")
    df = df.with_columns(pl.Series(label_name, y_arr))
    df.write_csv(path)
    return df

In [53]:
train_df = save_glove_csv(X_train_glove, y_train, "data/data-GloVe/train_glove.csv")
val_df = save_glove_csv(X_val_glove, y_val, "data/data-GloVe/val_glove.csv")

print(train_df.shape)
print(train_df.show())

(101225, 26)


dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,dim_11,dim_12,dim_13,dim_14,dim_15,dim_16,dim_17,dim_18,dim_19,dim_20,dim_21,dim_22,dim_23,dim_24,genre
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str
0.64515,-0.857273,-0.72595,-0.456712,0.759818,0.762,-0.831494,0.8591,0.797062,0.717474,-0.714412,-0.789534,-0.878479,-0.887971,0.721011,0.877996,0.852376,0.487466,-0.921126,0.837839,0.648419,0.889627,0.750056,0.847652,0.795449,"""rock"""
0.875382,-1.094005,-0.935994,-0.291348,1.083889,0.88823,-1.070388,1.005889,1.183798,1.04047,-0.96656,-1.033001,-1.176188,-1.118471,0.998465,1.051046,1.250318,1.211904,-1.068512,1.0786,0.651714,1.094056,1.134071,0.930314,1.381825,"""rock"""
0.887986,-1.047248,-0.941734,-0.371151,0.898243,0.735524,-1.011915,0.968973,1.031847,0.972615,-0.988913,-0.916275,-1.026412,-1.02347,0.962184,1.001094,1.038497,0.92168,-1.056259,1.008359,0.753951,1.027927,0.921138,0.992353,1.109551,"""rap"""
0.768788,-0.777757,-0.87642,-0.149712,0.73878,0.488544,-0.744638,0.855391,0.765491,0.776827,-0.785771,-0.739439,-0.723591,-0.850312,0.72762,0.764268,0.900733,0.96979,-0.86072,0.81342,0.586236,0.815459,0.78511,0.728314,1.044259,"""rap"""
0.616219,-0.831594,-0.722939,-0.018845,0.687747,0.450865,-0.808817,0.664147,0.888977,0.820259,-0.786522,-0.758755,-0.802129,-0.773624,0.825277,0.789861,0.970561,1.112917,-0.839097,0.782403,0.395153,0.832469,0.774726,0.699186,1.181321,"""rb"""


None
